# 19장 텍스트 데이터 — 글자를 숫자로

**3단계 · 다시 만들기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_3_build.ipynb)

이 노트북에는 **주석만 남긴 코드**가 들어 있습니다. 책과 1단계 노트북을 덮고, 주석을 실마리 삼아 코드를 되살려 보세요. 아래 표의 「다시 만들기」 과제가 이 노트북의 목표입니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 채운 뒤 셀을 실행해 저장하고, 확인 셀로 돌려 봅니다.
- 막히면 오류 메시지를 먼저 읽고(부록 A), 그래도 안 되면 1단계 노트북에서 그 부분만 확인하세요.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `fetch_statutes.py` 로 데이터를 받고 `p01`~`p03`, `statutes_dataset.py` 를 치고 실행 | [ch19_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_1_follow.ipynb) |
| 2. 주석 달기 | `Vocab.encode` 와 `BagOfWords.forward` 각 줄 뒤에 shape을 주석으로 | [ch19_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `StatuteDataset` 복원. 그다음 14장 패키지의 `data/` 에 `statutes.py` 로 넣고 `--dataset statutes --model bow` 로 학습되게 등록 | [ch19_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch19_3_build.ipynb) **(지금 이 노트북)** |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "19"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/19")

## `statutes_dataset.py` — 대상: `StatuteDataset`

In [ ]:
%%writefile statutes_dataset.py
# 19.4 텍스트 Dataset: 조문 → (토큰 번호 텐서, 법령 번호). 15장의 ImageFolderDataset 과 같은 뼈대.
import os
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader

DATA = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "statutes")
PAD, UNK = "<pad>", "<unk>"


def tokenize(s):
    return s.split()


class Vocab:
    def __init__(self, sentences, min_count=2, max_size=8000):
        counter = Counter(t for s in sentences for t in tokenize(s))
        self.itos = [PAD, UNK] + [t for t, c in counter.most_common(max_size) if c >= min_count]
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, s, max_len):
        ids = [self.stoi.get(t, 1) for t in tokenize(s)][:max_len]
        return ids + [0] * (max_len - len(ids))


class StatuteDataset(Dataset):
    """여러 법령의 조문을 모아 '어느 법령의 조문인가' 를 맞히는 분류 데이터셋."""

    def __init__(self, laws, split="train", vocab=None, max_len=40, test_ratio=0.2, seed=0):
        # '제786조 삭제' 같은 빈 조문은 뺀다
        # 같은 씨앗이면 같은 분할
        # 어휘는 학습 데이터로만 만든다
        raise NotImplementedError("여기를 채우세요")

    def __len__(self):
        raise NotImplementedError("여기를 채우세요")

    def __getitem__(self, i):
        raise NotImplementedError("여기를 채우세요")


if __name__ == "__main__":
    laws = ["민법", "형법", "민사소송법", "형사소송법"]
    train = StatuteDataset(laws, "train")
    test = StatuteDataset(laws, "test", vocab=train.vocab)        # 시험 데이터는 학습 어휘를 그대로 쓴다
    print("학습", len(train), "시험", len(test), "| 어휘", len(train.vocab), "| 클래스", train.classes)
    x, y = train[0]
    print("train[0] ->", tuple(x.shape), x.dtype, "| label", y, train.classes[y])
    print("  원문:", train.items[0][0][:50])
    print("  복원:", " ".join(train.vocab.itos[i] for i in x.tolist() if i != 0)[:50])
    xb, yb = next(iter(DataLoader(train, batch_size=8, shuffle=True)))
    print("배치:", tuple(xb.shape), tuple(yb.shape))
    counts = Counter(train.classes[l] for _, l in train.items)
    print("학습 클래스별:", dict(counts))

### 확인

In [ ]:
!python statutes_dataset.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- statutes_dataset.py

## 그다음

14장 패키지의 `data/` 에 `statutes.py` 로 넣고 `--dataset statutes --model bow` 로 학습되게 등록하세요. 이 과제는 14장 패키지(`code/14`)를 고치는 일입니다. 왼쪽 폴더 아이콘에서 `code/14` 의 파일을 열어 고치거나, 셀 첫 줄에 `%%writefile ../14/networks/새파일.py` 처럼 적어 저장한 뒤 아래 셀로 확인하세요.

In [ ]:
# 여기에 이어서 작성하세요